# Calibration-window repeat — Sampler

Copy of `benchmark-FULLCHAIN_-_SAMPLER_JOBID_WORKFLOW_RND_nonzero_fixed.ipynb`, retaining the original cell order, authentication, circuit builders, provider options/order, and result formats. Original outputs are cleared.

Changes: selected subset, per-window output folders, and submission guards. Enrich results later using `retrieve_ibm_job_metadata.ipynb`; no new enrichment code is included here.

Subset: BV-50, QPE-20, GHZ-50, RND-50. Submission and collection are initially disabled. To submit, set `SUBMIT_JOBS=True` and `COLLECT_RESULTS_FROM_JOB_IDS=False`. To collect later, set `SUBMIT_JOBS=False` and `COLLECT_RESULTS_FROM_JOB_IDS=True`; keep the same labels and `REBUILD_WORKLOAD_CACHE=False`.

Run the subset once per calibration window, using matching `WINDOW_ID` in both copies. Different labels create a new submission destination; do not change them to bypass an interrupted run.

A `.submission_started` marker blocks repeat submission to the same destination. After an interruption, **check IBM for already submitted jobs before retrying**. Do not delete the marker until reconciled. This is a safety stop, not automatic recovery. Estimator still waits for results, as in the original.

See `CALIBRATION_REPEAT_PLAN.md` for the proposed 2–3-window protocol. Labels alone do not verify distinct calibrations. Preparing these copies has not submitted jobs.



# Sampler Benchmark: BV, QPE, Native RND, and GHZ

This notebook is the **Sampler version** of the benchmarking workflow. It keeps the same overall pattern as the Estimator notebook:

1. configure IBM Quantum / backend / pass manager,
2. construct reproducible workloads for each selected system size,
3. submit the same workloads to each provider,
4. save per-job JSON and CSV rows after every size, and
5. summarize / plot the success probability and QPU usage.

The Sampler benchmark uses circuits whose ideal sampled outputs are known. For BV, QPE, and native RND, the main metric is exact-output probability. For RND, the target is a nonzero balanced initial bitstring rather than `0...0`, so relaxation to zero is penalized instead of rewarded:

```text
success_probability = counts[expected_bitstring] / total_shots
```

For GHZ, the main metric is valid-output probability:

```text
success_probability = (counts["0...0"] + counts["1...1"]) / total_shots
```

The workload map is:

```python
algorithm_n_values = {
    "BV":  [50],
    "QPE": [20],
    "RND": [50],
    "GHZ": [50],
}
```

There is **no QESEM block** here because this is a Sampler primitive workflow. The Q-CTRL Performance Management block uses `primitive="sampler"` when enabled.


In [ ]:

# Import packages.

import os
import json
import pickle
import traceback
import random
from pathlib import Path
from math import pi
from fractions import Fraction
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

import qiskit
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime.options import SamplerOptions
from qiskit_ibm_catalog import QiskitFunctionsCatalog



## Configuration

`n` is the number of **measured output bits**.  BV and QPE each use one extra target/ancilla qubit, so their physical circuit width is `n + 1`.  With `n = 125`, those circuits require 126 qubits.

The random inverse circuit is deliberately kept at `rnd_depth = 5`.  A depth equal to `n` at 125 qubits would create a very large arbitrary-unitary workload and can become expensive to transpile and run.


Use `enabled_algorithms` to turn individual workloads on/off during debugging or cost-control runs.

In [ ]:
# -----------------------------
# User-facing benchmark settings
# -----------------------------

# Run the subset ONCE per calibration window; change WINDOW_ID manually.
# Use matching labels in both copies. Labels alone do not verify calibration.
WINDOW_ID = "w01"

RESULT_DIR = Path("results/calibration_repeats") / WINDOW_ID / "sampler"
PLOT_DIR = RESULT_DIR / "plots"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

backend_name = "ibm_pittsburgh"
optimization_level = 3
shot_count = 2**15

# Algorithm-specific measured-output widths.
# This avoids forcing deep algorithmic circuits to the same large widths as BV/GHZ.
algorithm_n_values = {
    "BV":  [50],
    "QPE": [20],
    "RND": [50],
    "GHZ": [50],
}

# Union of all requested widths, used to batch workloads by n.
n_samples = sorted({n for values in algorithm_n_values.values() for n in values})

master_seed = 42
qpe_phase_fraction = 0.25     # theta in exp(2*pi*i*theta); expected key is 0100... for this value.
rnd_depth = 4                 # Native mirror RND depth. Increase only after pilot runs show nonzero signal.
rnd_seed_base = 4242

run_ibm_raw = True
run_ibm_measure_twirling = True
run_qctrl = True

# -----------------------------
# Job workflow controls
# -----------------------------
# First pass:
#   SUBMIT_JOBS = True
#   COLLECT_RESULTS_FROM_JOB_IDS = False
#
# Later pass, after jobs finish:
#   SUBMIT_JOBS = False
#   COLLECT_RESULTS_FROM_JOB_IDS = True
#
# This separates submission from retrieval so the notebook does not block on job.result().
SUBMIT_JOBS = False
COLLECT_RESULTS_FROM_JOB_IDS = False

# Keep False for normal use so re-running the submission cell does not create duplicates.
ALLOW_RESUBMIT_EXISTING = False

# If True, the collection cell checks status and skips queued/running jobs instead of blocking.
# Set False only if you intentionally want job.result() to wait.
COLLECT_ONLY_COMPLETED = True

# Submission already builds the cache. Keep False when collecting the same jobs.
REBUILD_WORKLOAD_CACHE = False

JOB_MANIFEST_FILE = RESULT_DIR / "sampler_benchmark_submitted_jobs.json"
WORKLOAD_CACHE_FILE = RESULT_DIR / "sampler_benchmark_workloads_by_n.pkl"

enabled_algorithms = {
    "BV": True,
    "QPE": True,
    "RND": True,
    "GHZ": True,
}

# Gate twirling is intentionally not used in this notebook because the workload
# includes controlled-phase QPE gates and randomized mirror circuits.
# Measurement twirling is the safer Sampler-side comparison option.


In [ ]:

# Authenticate and load backend.

load_dotenv()

IBM_QUANTUM_TOKEN = os.getenv("IBM_QUANTUM_TOKEN_EVIDA")
IBM_QUANTUM_INSTANCE = os.getenv("IBM_QUANTUM_INSTANCE_CUA_EVIDA")

if IBM_QUANTUM_TOKEN and IBM_QUANTUM_INSTANCE:
    QiskitRuntimeService.save_account(
        token=IBM_QUANTUM_TOKEN,
        instance=IBM_QUANTUM_INSTANCE,
        overwrite=True,
    )
else:
    print(
        "IBM_QUANTUM_TOKEN_CUA and/or IBM_QUANTUM_INSTANCE_CUA_OPEN were not found. "
        "Using the default saved IBM Quantum account, if available."
    )

service = QiskitRuntimeService()
backends = service.backends()

print(f"Account OK. {len(backends)} backend(s) available:")
for b in backends[:5]:
    print(f"  {b.name} ({b.num_qubits} qubits)")

backend = service.backend(backend_name)
print(f"\nSelected backend: {backend.name} ({backend.num_qubits} qubits)")

pm = generate_preset_pass_manager(
    optimization_level=optimization_level,
    backend=backend,
)

if run_qctrl or COLLECT_RESULTS_FROM_JOB_IDS:
    catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")
else:
    catalog = None

if run_qctrl:
    perf_mgmt = catalog.load("q-ctrl/performance-management")
else:
    perf_mgmt = None



## Circuit builders

The bitstring convention below is chosen to make scoring simple:

- `expected_bitstring` is written in the same left-to-right order used by Qiskit's counts keys.
- `measure(q[i], c[i])` means Qiskit displays `c[n-1] ... c[0]`, so basis-state preparation functions load `reversed(bitstring)` onto qubits.
- BV, QFT round-trip, QPE, and RND all have one dominant ideal output bitstring.
- RND uses a balanced nonzero initial bitstring as the expected output, not `0...0`.


For QPE, controlled-phase angles are reduced modulo `2*pi` before insertion, which avoids huge angles at large `n` while preserving the intended unitary.

In [ ]:

def random_bitstring(n, seed):
    """Return a reproducible n-bit string in Qiskit counts-key order."""
    rng = random.Random(seed)
    return "".join(rng.choice("01") for _ in range(n))


def assert_bitstring(bitstring, n=None):
    if not isinstance(bitstring, str) or any(bit not in "01" for bit in bitstring):
        raise ValueError("bitstring must be a string containing only '0' and '1'")
    if n is not None and len(bitstring) != n:
        raise ValueError(f"Expected a {n}-bit string, got {len(bitstring)} bits")


def create_bv_circuit(hidden_string):
    """
    Bernstein-Vazirani circuit.

    hidden_string is specified in Qiskit counts-key order. The oracle controls
    are therefore loaded with reversed(hidden_string), so the measured output key
    is directly comparable to hidden_string.
    """
    n = len(hidden_string)
    assert_bitstring(hidden_string, n)

    q = QuantumRegister(n + 1, "q")
    c = ClassicalRegister(n, "c")
    qc = QuantumCircuit(q, c, name=f"BV_{n}")

    ancilla = q[n]
    qc.x(ancilla)
    qc.h(q)

    for i, bit in enumerate(reversed(hidden_string)):
        if bit == "1":
            qc.cx(q[i], ancilla)

    qc.h(q[:n])
    qc.measure(q[:n], c)

    return qc


def inverse_quantum_fourier_transform(qc, number_of_qubits):
    """Apply an inverse QFT to qubits 0..number_of_qubits-1 in-place."""
    for qubit in range(number_of_qubits // 2):
        qc.swap(qubit, number_of_qubits - qubit - 1)

    for j in range(number_of_qubits):
        for control in range(j):
            qc.cp(-np.pi / float(2 ** (j - control)), control, j)
        qc.h(j)

    return qc


def phase_fraction_to_fraction(phase_fraction):
    """Represent a decimal phase fraction exactly enough for large-n powers."""
    return Fraction(str(phase_fraction)).limit_denominator()


def expected_qpe_bitstring(phase_fraction, m):
    """
    Expected QPE counts key for an exactly representable phase_fraction.

    For phase_fraction=0.25 and m>=2 this returns '0100...0'.
    """
    phase = phase_fraction_to_fraction(phase_fraction)
    scaled = phase * (1 << m)
    k = int(scaled + Fraction(1, 2)) % (1 << m)
    return format(k, f"0{m}b")


def create_qpe_circuit(phase_fraction, m):
    """
    Quantum phase estimation for U|1> = exp(2*pi*i*phase_fraction)|1>.

    m is the number of measured counting qubits. The circuit uses m+1 physical
    qubits because of the target eigenstate qubit.
    """
    q = QuantumRegister(m + 1, "q")
    c = ClassicalRegister(m, "c")
    qc = QuantumCircuit(q, c, name=f"QPE_{m}")

    target = q[m]
    qc.x(target)

    for j in range(m):
        qc.h(q[j])

    # Work in exact turns and reduce modulo 1 before converting to radians.
    # This avoids enormous angles such as 2*pi*0.25*2**124.
    phase = phase_fraction_to_fraction(phase_fraction)
    for j in range(m):
        turns = (phase * (1 << j)) % 1
        if turns != 0:
            qc.cp(2 * pi * float(turns), q[j], target)

    qc.barrier()
    inverse_quantum_fourier_transform(qc, m)
    qc.barrier()
    qc.measure(q[:m], c)

    return qc


def create_native_rnd_mirror_circuit(
    n,
    depth=4,
    seed=123,
    entanglement="linear",
    include_barriers=True,
    initial_state="balanced_random",
    return_target=False,
):
    """
    Native-gate randomized mirror circuit for Sampler benchmarking.

    The circuit first prepares a nonzero computational-basis state, then applies
    random native one-qubit gates and CZ entangling layers, and finally applies
    the exact inverse of only the random mirror body.

    Ideally, the measured output is the prepared initial bitstring. This avoids
    using 0...0 as the target, because T1 relaxation / amplitude damping can
    artificially increase the apparent success probability for an all-zero target.

    Parameters
    ----------
    n : int
        Number of measured qubits/bits.
    depth : int
        Number of random mirror layers before the inverse.
    seed : int
        Reproducibility seed. The random mirror body uses this seed; the initial
        bitstring uses a deterministic offset so that adding the nonzero target
        does not change the random mirror body generated by this seed.
    entanglement : {"linear", "random"}
        CZ entangling pattern.
    include_barriers : bool
        Insert barriers between layers.
    initial_state : {"balanced_random", "alternating", "random_nonzero", "all_ones"} or bitstring
        Initial computational-basis state. Explicit bitstrings are interpreted
        in Qiskit counts-key order, i.e., c[n-1]...c[0].
    return_target : bool
        If True, return (circuit, target_counts_key, target_q_order). If False,
        return only the circuit, so direct uses of this function remain compatible
        with code that expects a QuantumCircuit.
    """
    if n <= 0:
        raise ValueError("n must be a positive integer")

    q = QuantumRegister(n, "q")
    c = ClassicalRegister(n, "c")
    qc = QuantumCircuit(q, c, name=f"Native_RND_mirror_{n}_d{depth}")

    # Separate RNG streams:
    # - rng controls the random mirror body
    # - init_rng controls the nonzero input state
    #
    # This keeps the random mirror body stable for a given seed even after adding
    # the nonzero initial-state preparation.
    rng = np.random.default_rng(seed)
    init_seed = None if seed is None else int(seed) + 1_000_003
    init_rng = np.random.default_rng(init_seed)

    # Build the target bitstring in Qiskit counts-key order: c[n-1]...c[0].
    if initial_state == "balanced_random":
        target_bits = np.zeros(n, dtype=int)
        number_of_ones = max(1, n // 2)
        one_positions = init_rng.choice(n, size=number_of_ones, replace=False)
        target_bits[one_positions] = 1
        target_counts_key = "".join(str(bit) for bit in target_bits)

    elif initial_state == "alternating":
        target_counts_key = "".join("1" if i % 2 == 0 else "0" for i in range(n))

    elif initial_state == "random_nonzero":
        target_bits = init_rng.integers(0, 2, size=n, dtype=int)
        if int(target_bits.sum()) == 0:
            target_bits[int(init_rng.integers(0, n))] = 1
        target_counts_key = "".join(str(bit) for bit in target_bits)

    elif initial_state == "all_ones":
        target_counts_key = "1" * n

    elif isinstance(initial_state, str):
        if len(initial_state) != n or any(bit not in "01" for bit in initial_state):
            raise ValueError(
                "initial_state must be one of 'balanced_random', 'alternating', "
                "'random_nonzero', 'all_ones', or an explicit bitstring of length n."
            )
        target_counts_key = initial_state

    else:
        raise ValueError(
            "initial_state must be one of 'balanced_random', 'alternating', "
            "'random_nonzero', 'all_ones', or an explicit bitstring of length n."
        )

    # Since q[i] is measured into c[i], Qiskit count keys are displayed as
    # c[n-1]...c[0]. Therefore the qubit-preparation order is the reverse of
    # the target counts key.
    target_q_order = target_counts_key[::-1]

    # Prepare the nonzero initial basis state.
    for qubit, bit in enumerate(target_q_order):
        if bit == "1":
            qc.x(q[qubit])

    if include_barriers:
        qc.barrier()

    forward_layers = []
    one_qubit_gate_pool = ["x", "sx", "rz_pi_2", "rz_minus_pi_2"]

    for layer in range(depth):
        layer_ops = []

        # Random one-qubit layer.
        for qubit in range(n):
            gate = str(rng.choice(one_qubit_gate_pool))

            if gate == "x":
                qc.x(q[qubit])
            elif gate == "sx":
                qc.sx(q[qubit])
            elif gate == "rz_pi_2":
                qc.rz(np.pi / 2, q[qubit])
            elif gate == "rz_minus_pi_2":
                qc.rz(-np.pi / 2, q[qubit])
            else:
                raise ValueError(f"Unknown one-qubit gate: {gate}")

            layer_ops.append(("1q", gate, qubit))

        if include_barriers:
            qc.barrier()

        # Entangling layer.
        if entanglement == "linear":
            start = layer % 2
            entangling_pairs = [(qubit, qubit + 1) for qubit in range(start, n - 1, 2)]
        elif entanglement == "random":
            perm = rng.permutation(n)
            entangling_pairs = [
                (int(perm[i]), int(perm[i + 1]))
                for i in range(0, n - 1, 2)
            ]
        else:
            raise ValueError("entanglement must be 'linear' or 'random'")

        for q1, q2 in entangling_pairs:
            qc.cz(q[q1], q[q2])
            layer_ops.append(("2q", "cz", q1, q2))

        if include_barriers:
            qc.barrier()

        forward_layers.append(layer_ops)

    # Exact inverse of the random mirror body only.
    # Do not invert the initial-state preparation; that is the target output.
    for layer_ops in reversed(forward_layers):
        for op in reversed(layer_ops):
            if op[0] == "2q":
                _, gate, q1, q2 = op
                if gate == "cz":
                    qc.cz(q[q1], q[q2])
                else:
                    raise ValueError(f"Unknown two-qubit gate: {gate}")

            elif op[0] == "1q":
                _, gate, qubit = op

                if gate == "x":
                    qc.x(q[qubit])
                elif gate == "sx":
                    qc.sxdg(q[qubit])
                elif gate == "rz_pi_2":
                    qc.rz(-np.pi / 2, q[qubit])
                elif gate == "rz_minus_pi_2":
                    qc.rz(np.pi / 2, q[qubit])
                else:
                    raise ValueError(f"Unknown one-qubit gate: {gate}")

        if include_barriers:
            qc.barrier()

    qc.measure(q, c)

    if return_target:
        return qc, target_counts_key, target_q_order

    return qc

def create_ghz_circuit(n):
    """
    GHZ / cat-state circuit.

    The valid ideal output keys are 0...0 and 1...1.
    """
    q = QuantumRegister(n, "q")
    c = ClassicalRegister(n, "c")
    qc = QuantumCircuit(q, c, name=f"GHZ_{n}")

    qc.h(q[0])
    for i in range(n - 1):
        qc.cx(q[i], q[i + 1])

    qc.measure(q, c)
    return qc


In [ ]:

def algorithm_is_enabled_for_n(algorithm, n):
    """Return True when algorithm is enabled and n is in its requested width list."""
    return bool(enabled_algorithms.get(algorithm, True)) and n in algorithm_n_values.get(algorithm, [])


def make_sampler_workloads_for_n(n):
    """Build all Sampler benchmark circuits requested for one measured width n."""
    workloads = []

    if algorithm_is_enabled_for_n("BV", n):
        bv_hidden = random_bitstring(n, master_seed + 10_000 + n)
        workloads.append(
            {
                "algorithm": "BV",
                "n": n,
                "expected_bitstring": bv_hidden,
                "valid_bitstrings": [bv_hidden],
                "score_mode": "exact",
                "circuit": create_bv_circuit(bv_hidden),
                "notes": "Bernstein-Vazirani hidden string",
            }
        )

    if algorithm_is_enabled_for_n("QPE", n):
        qpe_expected = expected_qpe_bitstring(qpe_phase_fraction, n)
        workloads.append(
            {
                "algorithm": "QPE",
                "n": n,
                "expected_bitstring": qpe_expected,
                "valid_bitstrings": [qpe_expected],
                "score_mode": "exact",
                "circuit": create_qpe_circuit(qpe_phase_fraction, n),
                "notes": f"QPE phase_fraction={qpe_phase_fraction}",
            }
        )

    if algorithm_is_enabled_for_n("RND", n):
        rnd_circuit, rnd_expected, rnd_expected_q_order = create_native_rnd_mirror_circuit(
            n=n,
            depth=rnd_depth,
            seed=rnd_seed_base + n,
            entanglement="linear",
            include_barriers=True,
            initial_state="balanced_random",
            return_target=True,
        )
        workloads.append(
            {
                "algorithm": "RND",
                "n": n,
                "expected_bitstring": rnd_expected,
                "expected_bitstring_q_order": rnd_expected_q_order,
                "valid_bitstrings": [rnd_expected],
                "score_mode": "exact",
                "circuit": rnd_circuit,
                "notes": (
                    f"Native randomized mirror circuit, depth={rnd_depth}. "
                    "Target is a balanced nonzero initial bitstring, not 0...0."
                ),
            }
        )

    if algorithm_is_enabled_for_n("GHZ", n):
        zero = "0" * n
        one = "1" * n
        workloads.append(
            {
                "algorithm": "GHZ",
                "n": n,
                "expected_bitstring": f"{zero} OR {one}",
                "valid_bitstrings": [zero, one],
                "score_mode": "valid_set",
                "circuit": create_ghz_circuit(n),
                "notes": "GHZ valid outputs are all-zeros or all-ones",
            }
        )

    return workloads


def circuit_metric_dict(circuit, prefix):
    ops = dict(circuit.count_ops())
    metric = {
        f"{prefix}_num_qubits": int(circuit.num_qubits),
        f"{prefix}_num_clbits": int(circuit.num_clbits),
        f"{prefix}_depth": int(circuit.depth() or 0),
        f"{prefix}_size": int(circuit.size() or 0),
        f"{prefix}_ops_json": json.dumps({str(k): int(v) for k, v in ops.items()}, sort_keys=True),
    }

    for op_name in ["cx", "ecr", "cz", "swap", "cp", "unitary", "measure", "barrier"]:
        metric[f"{prefix}_{op_name}_count"] = int(ops.get(op_name, 0))

    return metric


def check_backend_capacity(workloads, backend):
    if not workloads:
        return 0

    max_width = max(item["circuit"].num_qubits for item in workloads)
    if max_width > backend.num_qubits:
        raise ValueError(
            f"The largest circuit uses {max_width} qubits, but {backend.name} "
            f"has {backend.num_qubits} qubits. Choose a larger backend or reduce algorithm_n_values."
        )

    return max_width


preview_rows = []
for n in n_samples:
    workloads = make_sampler_workloads_for_n(n)
    if not workloads:
        print(f"No enabled workloads for n={n}; skipping preview row.")
        continue

    max_width = check_backend_capacity(workloads, backend)
    for item in workloads:
        row = {
            "algorithm": item["algorithm"],
            "n": n,
            "physical_qubits": item["circuit"].num_qubits,
            "score_mode": item["score_mode"],
            "expected_prefix": item["expected_bitstring"][:16],
            "expected_suffix": item["expected_bitstring"][-16:],
            "valid_outputs": json.dumps(item.get("valid_bitstrings", [])),
            "expected_q_order_prefix": item.get("expected_bitstring_q_order", "")[:16],
            "expected_q_order_suffix": item.get("expected_bitstring_q_order", "")[-16:],
            "notes": item["notes"],
        }
        row.update(circuit_metric_dict(item["circuit"], "logical"))
        preview_rows.append(row)

preview_df = pd.DataFrame(preview_rows)
preview_df



## Sampler options and result helpers

SamplerV2 returns sampled bitstrings/counts, so this notebook scores each provider by how often the expected deterministic bitstring appears.  For IBM Runtime, the two configurations are:

- **IBM raw**: no explicit twirling / no explicit DD.
- **IBM measurement twirling**: measurement twirling enabled, gate twirling disabled.

Gate twirling is left off for this combined benchmark because QFT/QPE/RND include gates outside the Clifford-entangler-only setting where gate twirling is the clean comparison.


In [ ]:

def safe_set_nested(obj, dotted_path, value, required=False):
    """Safely set nested option fields such as twirling.enable_measure."""
    parts = dotted_path.split(".")
    target = obj

    for part in parts[:-1]:
        if not hasattr(target, part):
            message = f"Option path missing: {dotted_path}"
            if required:
                raise AttributeError(message)
            print("Warning:", message)
            return False
        target = getattr(target, part)

    final_attr = parts[-1]
    if not hasattr(target, final_attr):
        message = f"Option field missing: {dotted_path}"
        if required:
            raise AttributeError(message)
        print("Warning:", message)
        return False

    setattr(target, final_attr, value)
    return True


def make_ibm_raw_sampler_options(shots):
    options = SamplerOptions()
    options.default_shots = shots

    safe_set_nested(options, "twirling.enable_gates", False)
    safe_set_nested(options, "twirling.enable_measure", False)
    safe_set_nested(options, "dynamical_decoupling.enable", False)

    return options


def make_ibm_measure_twirling_sampler_options(shots):
    options = SamplerOptions()
    options.default_shots = shots

    safe_set_nested(options, "twirling.enable_gates", False)
    safe_set_nested(options, "twirling.enable_measure", True)
    safe_set_nested(options, "dynamical_decoupling.enable", False)

    return options


def get_job_id(job):
    """Extract job ID from IBM Runtime or Qiskit Function / Serverless jobs."""
    for attr in ["job_id", "id"]:
        value = getattr(job, attr, None)

        if callable(value):
            try:
                return value()
            except TypeError:
                pass

        if value is not None and not callable(value):
            return str(value)

    for attr in ["_job_id", "_id"]:
        value = getattr(job, attr, None)
        if value is not None:
            return str(value)

    return None


def get_qpu_seconds(job, primitive_result=None):
    """
    Best-effort extraction of QPU usage time.

    Returns (qpu_seconds, qpu_time_source).  If unavailable, returns
    (np.nan, "unavailable").
    """
    try:
        usage = getattr(job, "usage_estimation", None)
        if callable(usage):
            usage = usage()
        if isinstance(usage, dict):
            if "quantum_seconds" in usage:
                return float(usage["quantum_seconds"]), "usage_estimation.quantum_seconds"
            if "seconds" in usage:
                return float(usage["seconds"]), "usage_estimation.seconds"
    except Exception:
        pass

    try:
        metrics = job.metrics()
        if isinstance(metrics, dict):
            usage = metrics.get("usage", None)
            if isinstance(usage, dict):
                if "quantum_seconds" in usage:
                    return float(usage["quantum_seconds"]), "metrics.usage.quantum_seconds"
                if "seconds" in usage:
                    return float(usage["seconds"]), "metrics.usage.seconds"
            if "usage" in metrics and isinstance(metrics["usage"], (int, float)):
                return float(metrics["usage"]), "metrics.usage"
    except Exception:
        pass

    try:
        usage = job.usage()
        if isinstance(usage, dict):
            if "quantum_seconds" in usage:
                return float(usage["quantum_seconds"]), "usage.quantum_seconds"
            if "seconds" in usage:
                return float(usage["seconds"]), "usage.seconds"
    except Exception:
        pass

    try:
        runtime_jobs = job.runtime_jobs()
        total_qpu_seconds = 0.0
        found_any = False
        for runtime_job in runtime_jobs:
            qpu_seconds, _ = get_qpu_seconds(runtime_job)
            if not np.isnan(qpu_seconds):
                total_qpu_seconds += qpu_seconds
                found_any = True
        if found_any:
            return total_qpu_seconds, "sum(runtime_jobs.qpu_seconds)"
    except Exception:
        pass

    try:
        if primitive_result is not None and hasattr(primitive_result, "metadata"):
            execution = primitive_result.metadata.get("execution", {})
            spans = execution.get("execution_spans", None)
            duration = getattr(spans, "duration", None)
            if duration is not None:
                return float(duration), "result.metadata.execution.execution_spans.duration"
    except Exception:
        pass

    return np.nan, "unavailable"


def to_jsonable(obj):
    """Convert numpy and Qiskit-ish values to JSON-safe objects."""
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, tuple):
        return [to_jsonable(v) for v in obj]
    return obj


def save_json(filepath, payload):
    filepath = Path(filepath)
    filepath.parent.mkdir(parents=True, exist_ok=True)
    with open(filepath, "w") as f:
        json.dump(to_jsonable(payload), f, indent=4)


def utc_now_string():
    return datetime.now(timezone.utc).isoformat()


def load_job_manifest(filepath=JOB_MANIFEST_FILE):
    filepath = Path(filepath)
    if not filepath.exists():
        return []
    with open(filepath) as f:
        data = json.load(f)
    if isinstance(data, dict) and "jobs" in data:
        return data["jobs"]
    if isinstance(data, list):
        return data
    raise ValueError(f"Unsupported manifest format in {filepath}")


def write_job_manifest(jobs, filepath=JOB_MANIFEST_FILE):
    filepath = Path(filepath)
    filepath.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "created_by": "benchmark-FULLCHAIN_-_SAMPLER job-id workflow",
        "updated_at_utc": utc_now_string(),
        "backend_name": backend_name,
        "shot_count": shot_count,
        "jobs": jobs,
    }
    with open(filepath, "w") as f:
        json.dump(to_jsonable(payload), f, indent=4)


def get_job_status(job):
    """Return a string status for IBM Runtime jobs and Qiskit Function jobs."""
    try:
        status = job.status()
    except Exception as exc:
        return f"STATUS_UNAVAILABLE: {exc}"

    # Runtime jobs often return a JobStatus enum; Functions often return a string.
    name = getattr(status, "name", None)
    if name:
        return str(name)
    return str(status)


def is_done_status(status):
    status_upper = str(status).upper()
    return status_upper in {"DONE", "COMPLETED", "SUCCESS", "SUCCEEDED"} or status_upper.endswith(".DONE")


def is_error_status(status):
    status_upper = str(status).upper()
    return any(token in status_upper for token in ["ERROR", "CANCEL", "FAILED"])


def submission_key(provider_name, n):
    return f"{provider_file_label(provider_name)}__n{n}"


def load_workload_cache(filepath=WORKLOAD_CACHE_FILE):
    filepath = Path(filepath)
    if not filepath.exists():
        return None
    with open(filepath, "rb") as f:
        return pickle.load(f)


def save_workload_cache(workloads_by_n, filepath=WORKLOAD_CACHE_FILE):
    filepath = Path(filepath)
    filepath.parent.mkdir(parents=True, exist_ok=True)
    with open(filepath, "wb") as f:
        pickle.dump(workloads_by_n, f)


def retrieve_job_from_id(job_id, job_family):
    """
    Retrieve a submitted job by ID.

    IBM Runtime Sampler jobs are retrieved with service.job(job_id).
    Qiskit Function jobs, including Q-CTRL Performance Management, are retrieved
    with catalog.get_job_by_id(job_id).
    """
    if job_family == "runtime":
        return service.job(job_id)

    if job_family == "qiskit_function":
        if catalog is None:
            raise RuntimeError(
                "catalog is None. Set COLLECT_RESULTS_FROM_JOB_IDS=True or run_qctrl=True "
                "before running the authentication cell."
            )
        return catalog.get_job_by_id(job_id)

    raise ValueError(f"Unknown job_family={job_family!r}")


def save_collection_error(submission, exc):
    provider_name = submission["provider"]
    n = submission["n"]
    error_payload = {
        "provider": provider_name,
        "n": n,
        "job_id": submission.get("job_id"),
        "job_family": submission.get("job_family"),
        "status": submission.get("last_status"),
        "error_type": type(exc).__name__,
        "error_message": str(exc),
        "traceback": traceback.format_exc(),
        "collected_at_utc": utc_now_string(),
    }
    error_path = RESULT_DIR / f"{provider_file_label(provider_name)}_SAMPLER_n{n}_ERROR.json"
    save_json(error_path, error_payload)
    return error_path


In [ ]:

def extract_counts_from_pub_result(pub_result):
    """
    Extract counts from a Sampler PubResult.

    Circuits in this notebook use a classical register named 'c', but the helper
    also checks common fallback register names and then any DataBin register with
    a get_counts() method.
    """
    data = pub_result.data

    for register_name in ["c", "meas", "c0"]:
        if hasattr(data, register_name):
            register = getattr(data, register_name)
            if hasattr(register, "get_counts"):
                return dict(register.get_counts()), register_name

    try:
        keys = list(data.keys())
    except Exception:
        keys = []

    for register_name in keys:
        register = getattr(data, register_name)
        if hasattr(register, "get_counts"):
            return dict(register.get_counts()), register_name

    raise AttributeError(
        "Could not find a classical register with get_counts() in pub_result.data. "
        f"Available fields: {keys}"
    )


def clean_counts_keys(counts, n_bits):
    """Normalize count keys by removing spaces and zero-padding if needed."""
    cleaned = {}

    for key, value in counts.items():
        bitstring = str(key).replace(" ", "")
        if len(bitstring) < n_bits:
            bitstring = bitstring.zfill(n_bits)
        cleaned[bitstring] = cleaned.get(bitstring, 0) + int(value)

    return dict(sorted(cleaned.items(), key=lambda item: item[1], reverse=True))


def top_k_counts(counts, k=25):
    return dict(sorted(counts.items(), key=lambda item: item[1], reverse=True)[:k])


def score_counts(counts, expected_bitstring, score_mode="exact", valid_bitstrings=None):
    """
    Score Sampler counts.

    exact:     success is counts[expected_bitstring] / shots.
    valid_set: success is sum(counts[key] for key in valid_bitstrings) / shots.
    """
    total_shots = int(sum(counts.values()))
    valid_bitstrings = list(valid_bitstrings or [expected_bitstring])

    if total_shots == 0:
        return {
            "total_shots": 0,
            "expected_count": 0,
            "success_probability": np.nan,
            "failure_probability": np.nan,
            "top_bitstring": None,
            "top_count": 0,
            "top_probability": np.nan,
            "correct_is_top": False,
            "score_mode": score_mode,
            "valid_count": 0,
            "valid_probability": np.nan,
            "zero_count": np.nan,
            "one_count": np.nan,
        }

    if score_mode == "exact":
        expected_count = int(counts.get(expected_bitstring, 0))
        valid_count = expected_count
    elif score_mode == "valid_set":
        valid_count = int(sum(counts.get(bitstring, 0) for bitstring in valid_bitstrings))
        expected_count = valid_count
    else:
        raise ValueError("score_mode must be 'exact' or 'valid_set'")

    top_bitstring, top_count = max(counts.items(), key=lambda item: item[1])
    success_probability = valid_count / total_shots

    zero_key = None
    one_key = None
    if valid_bitstrings:
        n_bits = len(valid_bitstrings[0])
        zero_key = "0" * n_bits
        one_key = "1" * n_bits

    return {
        "total_shots": total_shots,
        "expected_count": expected_count,
        "success_probability": success_probability,
        "failure_probability": 1.0 - success_probability,
        "top_bitstring": top_bitstring,
        "top_count": int(top_count),
        "top_probability": int(top_count) / total_shots,
        "correct_is_top": bool(top_bitstring in set(valid_bitstrings)),
        "score_mode": score_mode,
        "valid_count": valid_count,
        "valid_probability": success_probability,
        "zero_count": int(counts.get(zero_key, 0)) if zero_key is not None else np.nan,
        "one_count": int(counts.get(one_key, 0)) if one_key is not None else np.nan,
    }


def provider_file_label(provider_name):
    return (
        provider_name.upper()
        .replace("+", "PLUS")
        .replace("-", "_")
        .replace(" ", "_")
        .replace("/", "_")
    )


def collect_sampler_rows(provider_name, job, primitive_result, workloads):
    """Convert a Sampler job result into CSV rows and a JSON payload."""
    job_id = get_job_id(job)
    qpu_seconds, qpu_time_source = get_qpu_seconds(job, primitive_result)

    rows = []
    json_results = []

    for pub_index, (workload, pub_result) in enumerate(zip(workloads, primitive_result)):
        n = workload["n"]
        expected = workload["expected_bitstring"]
        valid_bitstrings = workload.get("valid_bitstrings", [expected])
        score_mode = workload.get("score_mode", "exact")

        raw_counts, register_name = extract_counts_from_pub_result(pub_result)
        counts = clean_counts_keys(raw_counts, n_bits=n)
        score = score_counts(
            counts,
            expected_bitstring=expected,
            score_mode=score_mode,
            valid_bitstrings=valid_bitstrings,
        )

        logical_metrics = circuit_metric_dict(workload["circuit"], "logical")
        isa_metrics = circuit_metric_dict(workload["isa_circuit"], "isa")

        row = {
            "provider": provider_name,
            "job_id": job_id,
            "qpu_seconds": qpu_seconds,
            "qpu_time_source": qpu_time_source,
            "pub_index": pub_index,
            "algorithm": workload["algorithm"],
            "n": n,
            "physical_qubits": workload["circuit"].num_qubits,
            "expected_bitstring": expected,
            "expected_prefix": expected[:24],
            "expected_suffix": expected[-24:],
            "valid_bitstrings_json": json.dumps(valid_bitstrings),
            "score_mode": score_mode,
            "classical_register": register_name,
            "notes": workload["notes"],
        }
        row.update(score)
        row.update(logical_metrics)
        row.update(isa_metrics)

        rows.append(row)

        json_results.append(
            {
                "pub_index": pub_index,
                "algorithm": workload["algorithm"],
                "n": n,
                "physical_qubits": workload["circuit"].num_qubits,
                "expected_bitstring": expected,
                "valid_bitstrings": valid_bitstrings,
                "score_mode": score_mode,
                "classical_register": register_name,
                "score": score,
                "top_25_counts": top_k_counts(counts, k=25),
                "all_counts": counts,
                "logical_metrics": logical_metrics,
                "isa_metrics": isa_metrics,
                "notes": workload["notes"],
            }
        )

    payload = {
        "provider": provider_name,
        "job_id": job_id,
        "qpu_seconds": qpu_seconds,
        "qpu_time_source": qpu_time_source,
        "backend_name": backend.name,
        "shot_count_requested": shot_count,
        "algorithm_n_values": algorithm_n_values,
        "results": json_results,
    }

    return rows, payload


## Submit all jobs first, then retrieve by job ID

This section has two independent modes.

1. **Submission run:** set `SUBMIT_JOBS = True` and `COLLECT_RESULTS_FROM_JOB_IDS = False`. The notebook submits every enabled provider/width job, saves the job IDs to `sampler_benchmark_submitted_jobs.json`, and stops without calling `job.result()`.

2. **Collection run:** later, set `SUBMIT_JOBS = False` and `COLLECT_RESULTS_FROM_JOB_IDS = True`. The notebook reloads the job IDs and retrieves finished results by job ID.

In [ ]:
# Stop Run All by default. Collection does not submit new jobs.
if not SUBMIT_JOBS and not COLLECT_RESULTS_FROM_JOB_IDS:
    raise RuntimeError("Choose submission OR collection in the settings before running this cell.")

if SUBMIT_JOBS:
    # Blocks a second submission, including after an interrupted run.
    # Reconcile existing jobs before removing this marker or changing labels.
    (RESULT_DIR / ".submission_started").open("x").close()

def prepare_sampler_workloads_by_n(force_rebuild=False):
    """
    Build and transpile all requested workloads, then cache them.

    The cache is important because result collection needs the exact workload order
    used at submission time in order to score each PubResult correctly.
    """
    if not force_rebuild:
        cached = load_workload_cache()
        if cached is not None:
            print(f"Loaded workload cache from {WORKLOAD_CACHE_FILE}")
            return cached

    workloads_by_n = {}

    for n in n_samples:
        print("=" * 80)
        print(f"Preparing Sampler benchmark workloads with measured n={n}")
        print("=" * 80)

        workloads = make_sampler_workloads_for_n(n)
        if not workloads:
            print(f"No enabled workloads for n={n}; skipping.")
            continue

        max_width = check_backend_capacity(workloads, backend)
        print(f"Largest physical circuit width for n={n}: {max_width} qubits")
        print("Algorithms:", ", ".join(workload["algorithm"] for workload in workloads))

        # IBM Runtime Sampler expects ISA-transpiled circuits.
        # Q-CTRL will use the original abstract circuits, but we still store ISA
        # circuits for comparable circuit metrics in the output table.
        for workload in workloads:
            print(f"Transpiling {workload['algorithm']} n={n}...")
            workload["isa_circuit"] = pm.run(workload["circuit"])
            print(
                f"  logical depth={workload['circuit'].depth()}, "
                f"ISA depth={workload['isa_circuit'].depth()}, "
                f"ISA size={workload['isa_circuit'].size()}"
            )

        workloads_by_n[n] = workloads

    save_workload_cache(workloads_by_n)
    print(f"Saved workload cache to {WORKLOAD_CACHE_FILE}")

    return workloads_by_n


workloads_by_n = prepare_sampler_workloads_by_n(force_rebuild=SUBMIT_JOBS or REBUILD_WORKLOAD_CACHE)


# ============================================================
# 1. Submit all jobs and save job IDs.
# ============================================================
submitted_jobs = load_job_manifest()
submitted_by_key = {
    item["submission_key"]: item
    for item in submitted_jobs
    if "submission_key" in item
}

if SUBMIT_JOBS:
    for n, workloads in workloads_by_n.items():
        print("\n" + "=" * 80)
        print(f"Submitting enabled providers for measured n={n}")
        print("=" * 80)

        pubs = [workload["isa_circuit"] for workload in workloads]
        algorithms = [workload["algorithm"] for workload in workloads]

        # ------------------------------------------------------------
        # IBM raw Sampler
        # ------------------------------------------------------------
        if run_ibm_raw:
            provider_name = "IBM raw"
            key = submission_key(provider_name, n)

            if key in submitted_by_key and not ALLOW_RESUBMIT_EXISTING:
                print(f"Skipping {provider_name} n={n}; existing job_id={submitted_by_key[key]['job_id']}")
            else:
                print(f"\nSubmitting {provider_name} Sampler job...")
                options = make_ibm_raw_sampler_options(shot_count)
                sampler = Sampler(mode=backend, options=options)
                job = sampler.run(pubs, shots=shot_count)
                job_id = get_job_id(job)
                status = get_job_status(job)
                print(f"{provider_name} job ID:", job_id)
                print(f"{provider_name} status:", status)

                submission = {
                    "submission_key": key,
                    "provider": provider_name,
                    "provider_file_label": provider_file_label(provider_name),
                    "n": n,
                    "job_id": job_id,
                    "job_family": "runtime",
                    "backend_name": backend.name,
                    "shot_count": shot_count,
                    "workload_count": len(workloads),
                    "algorithms": algorithms,
                    "result_json": str(RESULT_DIR / f"{provider_file_label(provider_name)}_SAMPLER_n{n}.json"),
                    "submitted_at_utc": utc_now_string(),
                    "status_at_submission": status,
                    "last_status": status,
                }
                submitted_jobs.append(submission)
                submitted_by_key[key] = submission
                write_job_manifest(submitted_jobs)
                print(f"Updated manifest: {JOB_MANIFEST_FILE}")

        # ------------------------------------------------------------
        # IBM measurement twirling Sampler
        # ------------------------------------------------------------
        if run_ibm_measure_twirling:
            provider_name = "IBM measurement twirling"
            key = submission_key(provider_name, n)

            if key in submitted_by_key and not ALLOW_RESUBMIT_EXISTING:
                print(f"Skipping {provider_name} n={n}; existing job_id={submitted_by_key[key]['job_id']}")
            else:
                print(f"\nSubmitting {provider_name} Sampler job...")
                options = make_ibm_measure_twirling_sampler_options(shot_count)
                sampler = Sampler(mode=backend, options=options)
                job = sampler.run(pubs, shots=shot_count)
                job_id = get_job_id(job)
                status = get_job_status(job)
                print(f"{provider_name} job ID:", job_id)
                print(f"{provider_name} status:", status)

                submission = {
                    "submission_key": key,
                    "provider": provider_name,
                    "provider_file_label": provider_file_label(provider_name),
                    "n": n,
                    "job_id": job_id,
                    "job_family": "runtime",
                    "backend_name": backend.name,
                    "shot_count": shot_count,
                    "workload_count": len(workloads),
                    "algorithms": algorithms,
                    "result_json": str(RESULT_DIR / f"{provider_file_label(provider_name)}_SAMPLER_n{n}.json"),
                    "submitted_at_utc": utc_now_string(),
                    "status_at_submission": status,
                    "last_status": status,
                }
                submitted_jobs.append(submission)
                submitted_by_key[key] = submission
                write_job_manifest(submitted_jobs)
                print(f"Updated manifest: {JOB_MANIFEST_FILE}")

        # ------------------------------------------------------------
        # Q-CTRL Performance Management Sampler
        # ------------------------------------------------------------
        if run_qctrl:
            provider_name = "Q-CTRL"
            key = submission_key(provider_name, n)

            if key in submitted_by_key and not ALLOW_RESUBMIT_EXISTING:
                print(f"Skipping {provider_name} n={n}; existing job_id={submitted_by_key[key]['job_id']}")
            else:
                print(f"\nSubmitting {provider_name} Performance Management Sampler job...")

                # Q-CTRL should receive abstract circuits, not ISA-transpiled circuits.
                qctrl_pubs = [(workload["circuit"],) for workload in workloads]

                job = perf_mgmt.run(
                    primitive="sampler",
                    pubs=qctrl_pubs,
                    backend_name=backend.name,
                    options={
                        "default_shots": shot_count,
                    },
                )
                job_id = get_job_id(job)
                status = get_job_status(job)
                print(f"{provider_name} job ID:", job_id)
                print(f"{provider_name} status:", status)

                submission = {
                    "submission_key": key,
                    "provider": provider_name,
                    "provider_file_label": provider_file_label(provider_name),
                    "n": n,
                    "job_id": job_id,
                    "job_family": "qiskit_function",
                    "backend_name": backend.name,
                    "shot_count": shot_count,
                    "workload_count": len(workloads),
                    "algorithms": algorithms,
                    "result_json": str(RESULT_DIR / f"{provider_file_label(provider_name)}_SAMPLER_n{n}.json"),
                    "submitted_at_utc": utc_now_string(),
                    "status_at_submission": status,
                    "last_status": status,
                }
                submitted_jobs.append(submission)
                submitted_by_key[key] = submission
                write_job_manifest(submitted_jobs)
                print(f"Updated manifest: {JOB_MANIFEST_FILE}")

    print("\nSubmission pass complete.")
    print(f"Saved {len(submitted_jobs)} submitted job record(s) to {JOB_MANIFEST_FILE}")
else:
    print("SUBMIT_JOBS is False; no new jobs submitted.")


# ============================================================
# 2. Retrieve completed results by job ID.
# ============================================================
sampler_rows = []

if COLLECT_RESULTS_FROM_JOB_IDS:
    submitted_jobs = load_job_manifest()
    if not submitted_jobs:
        raise RuntimeError(f"No submitted jobs found in {JOB_MANIFEST_FILE}")

    # In case results already exist from a previous partial collection, preserve them.
    existing_results_csv = RESULT_DIR / "sampler_benchmark_results.csv"
    if existing_results_csv.exists():
        existing_df = pd.read_csv(existing_results_csv)
        sampler_rows.extend(existing_df.to_dict(orient="records"))
        collected_job_ids = set(existing_df["job_id"].dropna().astype(str).unique())
        print(f"Loaded {len(existing_df)} existing result row(s) from {existing_results_csv}")
    else:
        collected_job_ids = set()

    for submission in submitted_jobs:
        provider_name = submission["provider"]
        n = int(submission["n"])
        job_id = str(submission["job_id"])
        job_family = submission["job_family"]

        print("\n" + "-" * 80)
        print(f"Checking {provider_name} n={n} job_id={job_id}")
        print("-" * 80)

        if job_id in collected_job_ids:
            print("Already collected in sampler_benchmark_results.csv; skipping.")
            continue

        try:
            job = retrieve_job_from_id(job_id, job_family)
            status = get_job_status(job)
            submission["last_status"] = status
            submission["last_checked_at_utc"] = utc_now_string()
            print("Status:", status)

            if is_error_status(status):
                raise RuntimeError(f"Job is in terminal non-success status: {status}")

            if COLLECT_ONLY_COMPLETED and not is_done_status(status):
                print("Job is not finished; skipping without blocking.")
                write_job_manifest(submitted_jobs)
                continue

            result = job.result()
            workloads = workloads_by_n[n]

            rows, payload = collect_sampler_rows(provider_name, job, result, workloads)
            sampler_rows.extend(rows)
            collected_job_ids.add(job_id)

            result_path = RESULT_DIR / f"{provider_file_label(provider_name)}_SAMPLER_n{n}.json"
            save_json(result_path, payload)

            sampler_df_partial = pd.DataFrame(sampler_rows)
            sampler_df_partial.to_csv(
                RESULT_DIR / "sampler_benchmark_results_partial.csv",
                index=False,
            )
            sampler_df_partial.to_csv(
                RESULT_DIR / "sampler_benchmark_results.csv",
                index=False,
            )

            submission["collected_at_utc"] = utc_now_string()
            submission["result_json"] = str(result_path)
            submission["collection_error"] = None
            write_job_manifest(submitted_jobs)

            print(f"Saved result JSON: {result_path}")
            print(f"Updated CSV: {RESULT_DIR / 'sampler_benchmark_results.csv'}")

        except Exception as exc:
            print(f"Collection failed for {provider_name} n={n}.")
            print(exc)
            error_path = save_collection_error(submission, exc)
            submission["collection_error"] = str(exc)
            submission["error_json"] = str(error_path)
            submission["last_checked_at_utc"] = utc_now_string()
            write_job_manifest(submitted_jobs)

    if sampler_rows:
        sampler_df = pd.DataFrame(sampler_rows)
        sampler_df.to_csv(RESULT_DIR / "sampler_benchmark_results.csv", index=False)
        print(f"\nCollection pass complete. Current rows: {len(sampler_df)}")
        display(sampler_df)
    else:
        print("\nNo results collected yet.")
else:
    print("COLLECT_RESULTS_FROM_JOB_IDS is False; no results retrieved.")



## Summary tables

The main score is the success probability of the expected bitstring.  The summary tables aggregate across algorithms and also keep a per-algorithm view.


In [ ]:
results_csv = RESULT_DIR / "sampler_benchmark_results.csv"
if not results_csv.exists():
    raise FileNotFoundError(
        f"{results_csv} does not exist yet. Submit jobs first, then set "
        "COLLECT_RESULTS_FROM_JOB_IDS=True after the jobs are DONE."
    )


sampler_df = pd.read_csv(results_csv)

summary_df = (
    sampler_df
    .groupby(["provider", "n"], as_index=False)
    .agg(
        job_id=("job_id", "first"),
        qpu_seconds=("qpu_seconds", "first"),
        qpu_time_source=("qpu_time_source", "first"),
        workload_count=("algorithm", "count"),
        algorithms=("algorithm", lambda values: ", ".join(sorted(set(values)))),
        mean_expected_count=("expected_count", "mean"),
        mean_success_probability=("success_probability", "mean"),
        min_success_probability=("success_probability", "min"),
        mean_top_probability=("top_probability", "mean"),
        fraction_correct_is_top=("correct_is_top", "mean"),
        mean_isa_depth=("isa_depth", "mean"),
        max_isa_depth=("isa_depth", "max"),
        mean_isa_cz_count=("isa_cz_count", "mean"),
        max_isa_cz_count=("isa_cz_count", "max"),
    )
)

algorithm_summary_df = (
    sampler_df
    .groupby(["provider", "algorithm", "n"], as_index=False)
    .agg(
        job_id=("job_id", "first"),
        qpu_seconds=("qpu_seconds", "first"),
        qpu_time_source=("qpu_time_source", "first"),
        score_mode=("score_mode", "first"),
        expected_count=("expected_count", "mean"),
        success_probability=("success_probability", "mean"),
        top_probability=("top_probability", "mean"),
        correct_is_top=("correct_is_top", "mean"),
        valid_count=("valid_count", "mean"),
        zero_count=("zero_count", "mean"),
        one_count=("one_count", "mean"),
        isa_depth=("isa_depth", "mean"),
        isa_size=("isa_size", "mean"),
        isa_cz_count=("isa_cz_count", "mean"),
    )
)

summary_df.to_csv(RESULT_DIR / "sampler_benchmark_summary.csv", index=False)
algorithm_summary_df.to_csv(RESULT_DIR / "sampler_benchmark_algorithm_summary.csv", index=False)

summary_df


In [ ]:

algorithm_summary_df



## Plots

These plots mirror the estimator notebook style: provider comparisons across `n`, QPU seconds, and per-algorithm success probability.


In [ ]:

provider_order = [
    provider for provider in [
        "IBM raw",
        "IBM measurement twirling",
        "Q-CTRL",
    ]
    if provider in sampler_df["provider"].unique()
]

algorithm_order = [algorithm for algorithm in ["BV", "QPE", "RND", "GHZ"] if algorithm in sampler_df["algorithm"].unique()]


def safe_filename(name):
    return "".join(ch if ch.isalnum() else "_" for ch in str(name)).strip("_")


def finite_ymax(plot_df, exclude_col="n", default=1.0):
    value_cols = [col for col in plot_df.columns if col != exclude_col]
    if not value_cols:
        return default
    values = plot_df[value_cols].to_numpy(dtype=float)
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return default
    return np.nanmax(finite)


def probability_ylim(plot_df):
    ymax = finite_ymax(plot_df, default=1.0)
    if not np.isfinite(ymax):
        return (0, 1.05)
    if ymax <= 0:
        return (0, 1e-3)
    return (0, min(1.05, max(1e-3, ymax * 1.25)))


def format_bar_label(value):
    if not np.isfinite(value):
        return ""
    if value == 0:
        return "0"
    if abs(value) < 1e-3:
        return f"{value:.2e}"
    return f"{value:.3f}"


def grouped_bar(plot_df, x_col, series_cols, title, ylabel, xlabel, filename, ylim=None):
    series_cols = [col for col in series_cols if col in plot_df.columns]
    x_labels = plot_df[x_col].astype(str).tolist()
    x = np.arange(len(plot_df))
    width = 0.8 / max(len(series_cols), 1)

    plt.figure(figsize=(10, 5))

    for i, col in enumerate(series_cols):
        offset = (i - (len(series_cols) - 1) / 2) * width
        y = plot_df[col].astype(float).values
        bars = plt.bar(x + offset, y, width, label=col)

        for bar, value in zip(bars, y):
            if np.isfinite(value):
                plt.text(
                    bar.get_x() + bar.get_width() / 2,
                    value,
                    format_bar_label(value),
                    ha="center",
                    va="bottom" if value >= 0 else "top",
                    fontsize=8,
                    rotation=90,
                )

    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.xticks(x, x_labels)

    if ylim is not None:
        plt.ylim(*ylim)

    plt.grid(axis="y", alpha=0.3)
    plt.legend()
    plt.tight_layout()

    filename = Path(filename)
    filename.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved plot:", filename)


def provider_pivot(df, value_col):
    plot_df = (
        df.pivot_table(
            index="n",
            columns="provider",
            values=value_col,
            aggfunc="mean",
        )
        .reset_index()
    )
    cols = ["n"] + [provider for provider in provider_order if provider in plot_df.columns]
    return plot_df[cols]


# Plot 1: mean success probability across algorithms available at each n.
plot_df = provider_pivot(summary_df, "mean_success_probability")
grouped_bar(
    plot_df=plot_df,
    x_col="n",
    series_cols=provider_order,
    title="Sampler benchmark: mean success probability across available algorithms",
    ylabel="Mean success probability",
    xlabel="Measured output bits n",
    filename=PLOT_DIR / "mean_success_probability.png",
    ylim=probability_ylim(plot_df),
)


# Plot 2: minimum success probability across algorithms available at each n.
plot_df = provider_pivot(summary_df, "min_success_probability")
grouped_bar(
    plot_df=plot_df,
    x_col="n",
    series_cols=provider_order,
    title="Sampler benchmark: minimum success probability across available algorithms",
    ylabel="Minimum success probability",
    xlabel="Measured output bits n",
    filename=PLOT_DIR / "min_success_probability.png",
    ylim=probability_ylim(plot_df),
)


# Plot 3: QPU seconds.
if "qpu_seconds" in summary_df.columns:
    qpu_df = summary_df.dropna(subset=["qpu_seconds"]).copy()
    if not qpu_df.empty:
        plot_df = provider_pivot(qpu_df, "qpu_seconds")
        ymax = finite_ymax(plot_df, default=1.0)
        grouped_bar(
            plot_df=plot_df,
            x_col="n",
            series_cols=provider_order,
            title="Sampler benchmark: QPU time",
            ylabel="QPU seconds",
            xlabel="Measured output bits n",
            filename=PLOT_DIR / "qpu_seconds.png",
            ylim=(0, max(1.0, ymax * 1.25)),
        )
    else:
        print("Skipping QPU time plot: qpu_seconds exists but all values are NaN.")
else:
    print("Skipping QPU time plot: qpu_seconds column not found.")


# Plot 4: per-algorithm success probability.
for algorithm in algorithm_order:
    subset = algorithm_summary_df[algorithm_summary_df["algorithm"] == algorithm].copy()
    if subset.empty:
        continue

    plot_df = provider_pivot(subset, "success_probability")
    metric_name = "Valid-output probability" if algorithm == "GHZ" else "Exact success probability"
    grouped_bar(
        plot_df=plot_df,
        x_col="n",
        series_cols=provider_order,
        title=f"Sampler benchmark: {algorithm} success probability",
        ylabel=metric_name,
        xlabel="Measured output bits n",
        filename=PLOT_DIR / f"{safe_filename(algorithm)}_success_probability.png",
        ylim=probability_ylim(plot_df),
    )


# Plot 5: ISA depth by algorithm.
for algorithm in algorithm_order:
    subset = algorithm_summary_df[algorithm_summary_df["algorithm"] == algorithm].copy()
    if subset.empty:
        continue

    plot_df = provider_pivot(subset, "isa_depth")
    ymax = finite_ymax(plot_df, default=1.0)
    grouped_bar(
        plot_df=plot_df,
        x_col="n",
        series_cols=provider_order,
        title=f"Sampler benchmark: {algorithm} ISA depth",
        ylabel="ISA depth",
        xlabel="Measured output bits n",
        filename=PLOT_DIR / f"{safe_filename(algorithm)}_isa_depth.png",
        ylim=(0, max(1.0, ymax * 1.25)),
    )


# Plot 6: ISA CZ count by algorithm.
for algorithm in algorithm_order:
    subset = algorithm_summary_df[algorithm_summary_df["algorithm"] == algorithm].copy()
    if subset.empty:
        continue

    plot_df = provider_pivot(subset, "isa_cz_count")
    ymax = finite_ymax(plot_df, default=1.0)
    grouped_bar(
        plot_df=plot_df,
        x_col="n",
        series_cols=provider_order,
        title=f"Sampler benchmark: {algorithm} ISA CZ count",
        ylabel="ISA CZ count",
        xlabel="Measured output bits n",
        filename=PLOT_DIR / f"{safe_filename(algorithm)}_isa_cz_count.png",
        ylim=(0, max(1.0, ymax * 1.25)),
    )

print("Done.")
print("Plots saved in:", PLOT_DIR)


## Copy IDs into the enrichment notebook

Run this cell after collecting results, with `RESULT_DIR` and `WINDOW_ID` still set to the same window. It reads the saved submission manifest (including jobs whose results are not yet collected), removes duplicate IDs, and prints the two input strings used by `retrieve_ibm_job_metadata.ipynb`. It makes no IBM calls and changes no files.

For multiple windows or both tracks, append each printed comment and ID list **inside** the corresponding existing triple-quoted string; do not replace IDs from earlier runs. Separate lists with a newline (the enrichment parser accepts newlines). Function IDs belong in `FUNCTION_JOB_IDS_TEXT`, not `BATCH_IDS_TEXT`.

Only saved IDs can be listed. After an interrupted execution, reconcile any jobs missing from the saved files with IBM before retrying; an empty or partial list is not proof that no jobs were submitted.


In [ ]:
# Local files only: no IBM calls and no submission.
import json

manifest_path = RESULT_DIR / "sampler_benchmark_submitted_jobs.json"
records = []
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    records = manifest["jobs"] if isinstance(manifest, dict) else manifest
families = {"runtime": "RUNTIME_JOB_IDS_TEXT", "qiskit_function": "FUNCTION_JOB_IDS_TEXT"}
family_field = "job_family"

ids_for_enrichment = {"RUNTIME_JOB_IDS_TEXT": [], "FUNCTION_JOB_IDS_TEXT": []}
for record in records:
    job_id = str(record.get("job_id") or "").strip()
    if not job_id:
        print("# WARNING: a saved record has no job_id; check the original output.")
        continue
    group = families[record[family_field]]  # Unknown families/providers must be reviewed.
    if job_id not in ids_for_enrichment[group]:
        ids_for_enrichment[group].append(job_id)

if not any(ids_for_enrichment.values()):
    print("# No saved job IDs found in", RESULT_DIR)
for variable, job_ids in ids_for_enrichment.items():
    print(f'{variable} = """')
    print(f"# {WINDOW_ID} / sampler")
    print(", ".join(job_ids))
    print('"""\n')
